# Антиспам на основе хакатона

Ражнов Андрей · МКБ251. Воспроизводим сохранённую модель писем; F1 турнира по URL к ней не относится.

In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd()
if not (ROOT/'artifacts').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from adapt_hackathon import read_emails, predict_bundle
import joblib, numpy as np
m=json.loads((ROOT/'artifacts/hackathon_v2/metrics.json').read_text())
print(json.dumps(m['basis'],ensure_ascii=False,indent=2))

{
  "repository": "https://github.com/Andrey123654/Hakaton",
  "commit": "afd2206d6d172b369b620d01ba10d6776219ad90",
  "file": "src/features.py",
  "sha256": "a66ad4d77f82d1b0369cbda4fa094765b07ef2046b1e3ceacfb3b26ec64f3089",
  "transfer": "24 offline URL features, max/mean aggregation plus count/truncation; OOF stacking",
  "not_transferred": "phishing weights, target labels, network snapshots, public F1"
}


## Единый тест и порог

Тестовые тексты удалены из development и validation. Срез 342 относится к исходному историческому пулу, а не к текущему обучению. Все показатели рассчитаны одной моделью.

In [2]:
print(json.dumps({k:m[k] for k in ['dataset','threshold','validation','test_full','test_unseen']},ensure_ascii=False,indent=2))

{
  "dataset": {
    "input_pool": 9055,
    "pool_after_test_purge": 7848,
    "development": 6278,
    "validation": 1570,
    "test": 1545,
    "normalized_test_overlap_purged": 1203,
    "test_unseen_relative_to_legacy_pool": 342,
    "conflicting_pool_texts_removed": 0,
    "sources_development": {
      "spamassassin": 3964,
      "lingspam": 2297,
      "synthetic": 17
    },
    "spambase_used_for_training": false
  },
  "threshold": 0.7103761388771067,
  "validation": {
    "threshold": 0.710376,
    "f1": 0.98204,
    "precision": 0.9791,
    "recall": 0.98498,
    "roc_auc": 0.9995,
    "pr_auc": 0.99811,
    "n": 1570,
    "spam": 333,
    "tp": 328,
    "fp": 7,
    "fn": 5,
    "tn": 1230,
    "false_positive_rate": 0.00566
  },
  "test_full": {
    "threshold": 0.710376,
    "f1": 0.93878,
    "precision": 0.99184,
    "recall": 0.8911,
    "roc_auc": 0.99211,
    "pr_auc": 0.9952,
    "n": 1545,
    "spam": 955,
    "tp": 851,
    "fp": 7,
    "fn": 104,
    "tn": 583,


In [3]:
df=read_emails(ROOT/'data/processed/test.csv')
b=joblib.load(ROOT/'artifacts/hackathon_v2/spam_bundle.joblib')
p=predict_bundle(b,df.iloc[:8])
print('Модель:',b['kind'])
print('Порог:',b['threshold'])
print('Первые оценки:',np.round(p,6).tolist())
assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()

Модель: hackathon-email-v2
Порог: 0.7103761388771067
Первые оценки: [0.001628, 0.000432, 0.000406, 1.4e-05, 0.001969, 0.036758, 5.7e-05, 0.00063]


## Ограничения

FPR на всём тесте 1,19 %, на исторически новом срезе — 7/24 = 29,17 %. Нужен свежий внешний тест. Категории размечены правилами. Отдельная абляция URL-признаков пока не выполнена. Полное обучение: `python -m src.adapt_hackathon`; проверка API: `python tools/verify_project.py`.